# Gainesville Traffic Crash Analysis (2015-2025)

**Course:** ISM 6423 - Data Analysis for Decision Support  
**Project:** Crash Busters  
**Dataset:** Gainesville Police Department traffic-crash records from the City of Gainesville Open Data Portal

## Project Objective
This notebook analyzes long-run crash patterns in Gainesville, Florida, with an emphasis on vulnerable road users, temporal risk windows, high-crash corridors, and factors associated with fatal outcomes. The workflow combines data cleaning, feature engineering, exploratory analysis, statistical hypothesis testing, logistic regression, and pre/post-COVID comparison.

> **Methodology note:** The dataset provides a reliable `Total Fatalities` field but does not contain a complete official injury-severity field. The notebook therefore uses a project-defined nonfatal severity proxy based on the number of people involved. Interpret that proxy as an analytical categorization, not an official injury classification.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import plotly.express as px
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import folium
from folium.plugins import HeatMap, MarkerCluster
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
import statsmodels.api as sm

## Exploratory Data Analysis (EDA) Results

### Overall Crash Trends
*   **Total Crashes Per Year**: There has been an overall decreasing trend in total crashes from 2015 to 2025.
*   **Fatal Crashes Per Year**: The number of fatal crashes per year has remained relatively stable over the study period (2015-2025).

### Crash Type Analysis
*   **Dominant Crash Type**: 'Motor Vehicle Only' crashes constitute the vast majority of incidents.
*   **Annual Frequency by Type**: 'Motor Vehicle Only' and 'Involving Cyclist' crashes have generally decreased over the years. In contrast, 'Involving Pedestrian' crashes show a slight increase in the later years of the study period.
*   **Severity by Crash Type**: There is a statistically significant association between Crash Type and Fatality (p-value < 0.05). Notably, crashes involving pedestrians and cyclists tend to have a higher proportion of fatal outcomes compared to motor vehicle only crashes.

### Time-Based Analysis
*   **Pre- vs. Post-COVID**: Comparing average annual crash rates from 2015-2019 (Pre-COVID) to 2020-2025 (Post-COVID):
    *   'Motor Vehicle Only' and 'Involving Cyclist' crashes saw a decrease in average annual rates.
    *   'Involving Pedestrian' crashes, however, experienced an increase in average annual rates.
*   **Crashes by Hour of Day**: Crash frequency exhibits a bimodal distribution, with peaks occurring during typical morning (8-9 AM) and afternoon/evening (3-6 PM) rush hours.
*   **Severity by Hour of Day**: A higher proportion of fatal crashes are observed during late night and early morning hours (e.g., 1-5 AM).

### Location-Based Analysis
*   **Top Crash Address Segments**: Specific road segments, such as 'SR 24 (SW ARCHER RD)', 'SW ARCHER RD', and 'SR 121 (SW 34TH ST)', consistently appear as high-crash locations.
*   **Intersection Type**: The majority of crashes (around 65%) occur 'NOT AT INTERSECTION', followed by 'FOUR-WAY INTERSECTION'.

In [ ]:
from pathlib import Path

DATA_CANDIDATES = [
    Path("../data/Traffic_Crashes_20260412.csv"),  # running from notebooks/
    Path("data/Traffic_Crashes_20260412.csv"),     # running from repository root
    Path("/content/Traffic_Crashes_20260412.csv"), # Google Colab fallback
]

FILE_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError(
        "Traffic_Crashes_20260412.csv was not found. "
        "Download the dataset as described in data/README.md and place it in the data/ folder."
    )

df_raw = pd.read_csv(FILE_PATH, low_memory=False)
print(f"Raw shape : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print("\nColumn names:")
print(df_raw.columns.tolist())

# Data types & missing values
print("\nData types:")
print(df_raw.dtypes)

print("\nMissing value count per column:")
missing = df_raw.isnull().sum()
print(missing[missing > 0])

Raw shape : 78,921 rows × 27 columns

Column names:
['Case Number', 'DHSMV Number', 'Crash Date', 'Crash Hour of Day', 'Crash minutes', 'Crash Day of Week', 'Address', 'Street Address number', 'Intersect Type', 'Distance', 'Street Direction', 'At/From Intersection', 'Total People ', 'Total Bycicles ', 'Total Pedestrians ', 'Total Vehicles ', 'Total Mopeds', 'Total Motorcycles', 'Total Buses ', 'Total Fatalities', 'Geox', 'Geoy', 'Longitude', 'Latitude', 'Location', 'City', 'State']

Data types:
Case Number                int64
DHSMV Number               int64
Crash Date                object
Crash Hour of Day          int64
Crash minutes              int64
Crash Day of Week         object
Address                   object
Street Address number     object
Intersect Type            object
Distance                  object
Street Direction          object
At/From Intersection      object
Total People               int64
Total Bycicles             int64
Total Pedestrians          int64
Total

In [ ]:
df1 = df_raw.copy()


df1.columns = df1.columns.str.strip()

#  Parse date
df1["Crash Date"] = pd.to_datetime(df1["Crash Date"], format="%m/%d/%Y", errors="coerce")

df1["Year"]    = df1["Crash Date"].dt.year
df1["Month"]   = df1["Crash Date"].dt.month
df1["Quarter"] = df1["Crash Date"].dt.quarter
df1["Week"]    = df1["Crash Date"].dt.isocalendar().week.astype(int)

# Build a readable Month-Year for plotting (e.g. "2020-01")
df1["YearMonth"] = df1["Crash Date"].dt.to_period("M").astype(str)


## Define the Study Window

In [ ]:
# Study window

df_study  = df1[df1["Year"].between(2015, 2025)].copy()
df_2026   = df1[df1["Year"] == 2026].copy()

print(f"Study window (2015-2025) : {len(df_study):,} crashes")
print(f"2026 partial data        : {len(df_2026):,} crashes")

Study window (2015-2025) : 58,032 crashes
2026 partial data        : 390 crashes


## Feature Engineering: Crash Type

In [ ]:
# Crash-type classification

def classify_crash(row):
    # Determine if the crash involves a bicycle or pedestrian
    has_bike  = row["Total Bycicles"]    >= 1
    has_ped   = row["Total Pedestrians"] >= 1

    # Classify the crash type based on involvement of vulnerable road users
    if has_bike and has_ped:
        return "Multi-Vulnerable" # Involves both cyclists and pedestrians
    elif has_bike:
        return "Involving Cyclist" # Involves cyclists only
    elif has_ped:
        return "Involving Pedestrian" # Involves pedestrians only
    else:
        return "Motor Vehicle Only" # Only motor vehicles involved

# Apply the classification function to create the 'Crash Type' column
df_study["Crash Type"] = df_study.apply(classify_crash, axis=1)


In [ ]:
df_study["Crash Type"].value_counts()

,count
Crash Type,
Motor Vehicle Only,55562
Involving Pedestrian,1297
Involving Cyclist,1171
Multi-Vulnerable,2


In [ ]:
df_study

,Case Number,DHSMV Number,Crash Date,Crash Hour of Day,Crash minutes,Crash Day of Week,Address,Street Address number,Intersect Type,Distance,...,Latitude,Location,City,State,Year,Month,Quarter,Week,YearMonth,Crash Type
0,223008254,25598677,2023-05-23,17,2,Tuesday,CLARK BUTLER BLVD,3101,NOT AT INTERSECTION,NaN,...,29.625609,POINT (-82.32272 29.65198),Gainesville,Florida,2023,5,2,21,2023-05,Motor Vehicle Only
1,223008252,25598681,2023-05-23,16,24,Tuesday,OLD ARCHER RD,NaN,T-INTERSECTION,NaN,...,29.651980,POINT (-82.32272 29.65198),Gainesville,Florida,2023,5,2,21,2023-05,Motor Vehicle Only
2,223008301,25598687,2023-05-24,7,23,Wednesday,NW 62ND AVE,NaN,FOUR-WAY INTERSECTION,NaN,...,29.651980,POINT (-82.32272 29.65198),Gainesville,Florida,2023,5,2,21,2023-05,Motor Vehicle Only
3,223008304,25598689,2023-05-24,14,18,Wednesday,SR 20 (NW 6TH ST),NaN,FOUR-WAY INTERSECTION,5 FEET,...,29.651980,POINT (-82.32272 29.65198),Gainesville,Florida,2023,5,2,21,2023-05,Motor Vehicle Only
4,223008253,25598682,2023-05-23,16,43,Tuesday,SR 20 (NW 6TH ST),NaN,T-INTERSECTION,NaN,...,29.651980,POINT (-82.32272 29.65198),Gainesville,Florida,2023,5,2,21,2023-05,Motor Vehicle Only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78527,225019355,27324834,2025-12-26,20,0,Friday,NE 49TH AVE,4640,NOT AT INTERSECTION,NaN,...,29.700800,POINT (-82.32265 29.65198),Gainesville,Florida,2025,12,4,52,2025-12,Motor Vehicle Only
78528,225019226,27324889,2025-12-26,0,0,Friday,NW 55TH BLVD,2056,OTHER (EXPLAIN IN NARRATIVE),FEET,...,29.704590,POINT (-82.32265 29.65198),Gainesville,Florida,2025,12,4,52,2025-12,Motor Vehicle Only
78529,225019306,27324824,2025-12-28,14,56,Sunday,SW 20TH AVE,NaN,FOUR-WAY INTERSECTION,40 FEET,...,29.651980,POINT (-82.32265 29.65198),Gainesville,Florida,2025,12,4,52,2025-12,Motor Vehicle Only
78530,225019264,27324820,2025-12-26,11,28,Friday,SW 34TH ST,1714,NOT AT INTERSECTION,NaN,...,29.636680,POINT (-82.32265 29.65198),Gainesville,Florida,2025,12,4,52,2025-12,Motor Vehicle Only


## Feature Engineering: Severity Proxy

In [ ]:
# Define a function to classify the severity of a crash
def classify_severity(row):
    # If there is at least one fatality, classify as 'Fatal'
    if row["Total Fatalities"] >= 1:
        return "Fatal"
    # If there are 2 or more people involved, it suggests a probable injury
    elif row["Total People"] >= 2:
        return "Probable Injury"
    # Otherwise, if no fatalities and less than 2 people, classify as 'Property Damage Only'
    else:
        return "Probable Property Damage"

# Apply the severity classification function to each row of the DataFrame
df_study["Severity"] = df_study.apply(classify_severity, axis=1)

## Feature Engineering: Time-of-Day Bins

In [ ]:
#  Time-of-day bins
def time_of_day(hour):
    if   6  <= hour < 12: return "Morning (6-11)"
    elif 12 <= hour < 17: return "Afternoon (12-16)"
    elif 17 <= hour < 21: return "Evening (17-20)"
    else:                  return "Night (21-5)"

df_study["Time of Day"] = df_study["Crash Hour of Day"].apply(time_of_day)

In [ ]:
df_study["Period"] = df_study["Year"].apply(
    lambda y: "Period 1: 2015–2019" if y <= 2019 else "Period 2: 2020–2025"
)

print("\n Feature engineering complete.")
print("\nCrash Type distribution (2015-2025):")
crash_type_distribution = df_study["Crash Type"].value_counts().reset_index()
crash_type_distribution.columns = ['Crash Type', 'Count']
print(crash_type_distribution)
print("\nSeverity distribution (2015-2025):")
print(df_study["Severity"].value_counts())



 Feature engineering complete.

Crash Type distribution (2015-2025):
             Crash Type  Count
0    Motor Vehicle Only  55562
1  Involving Pedestrian   1297
2     Involving Cyclist   1171
3      Multi-Vulnerable      2

Severity distribution (2015-2025):
Severity
Probable Injury             51404
Probable Property Damage     6478
Fatal                         150
Name: count, dtype: int64


## Annual Crash Trends by Severity and Road-User Involvement

In [ ]:
# Group by 'Year' and count the number of crashes
crashes_per_year = df_study.groupby('Year').size().reset_index(name='Total Crashes')

# Create an interactive scatter chart with a trend line using Plotly Express
# px.scatter automatically includes markers. We'll add lines connecting them later.
fig = px.scatter(crashes_per_year,
                x='Year',
                y='Total Crashes',
                title='Total Crashes Per Year (2015-2025) with Trendline',
                trendline='ols',
                trendline_color_override='red',
                hover_data={'Total Crashes': ':,0f'})

# Update the scatter trace to show lines and markers.
# The 'selector' is used to target only the main scatter plot data (named 'Total Crashes')
# and not the trendline, ensuring the trendline remains a simple line.
fig.update_traces(mode='lines+markers', selector=dict(name='Total Crashes'))

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Total Crashes',
    hovermode='x unified'
)

fig.show()


In [ ]:
# Filter for fatal crashes
df_fatal = df_study[df_study['Severity'] == 'Fatal'].copy()

# Group by 'Year' and count the number of fatal crashes
fatal_crashes_per_year = df_fatal.groupby('Year').size().reset_index(name='Total Fatal Crashes')

# Create an interactive scatter chart with a trend line for fatal crashes
fig_fatal = px.scatter(fatal_crashes_per_year,
                     x='Year',
                     y='Total Fatal Crashes',
                     title='Fatal Crashes Per Year (2015-2025) with Trendline',
                     trendline='ols',
                     trendline_color_override='red',
                     hover_data={'Total Fatal Crashes': ':,0f'})

# Update the scatter trace to show lines and markers for fatal crashes
fig_fatal.update_traces(mode='lines+markers', selector=dict(name='Total Fatal Crashes'))

fig_fatal.update_layout(
    xaxis_title='Year',
    yaxis_title='Total Fatal Crashes',
    hovermode='x unified'
)

fig_fatal.show()

In [ ]:
freq_by_type = (
    df_study
    .groupby(["Year", "Crash Type"])["Case Number"]
    .count()
    .unstack("Crash Type")
    .fillna(0)
    .astype(int)
    .reset_index()
)

crash_type_cols = ["Motor Vehicle Only",
                   "Involving Cyclist",
                   "Involving Pedestrian",
                   "Multi-Vulnerable"]

# Melt the DataFrame to long format for Plotly Express
df_melted = freq_by_type.melt(id_vars=['Year'], value_vars=crash_type_cols, var_name='Crash Type', value_name='Count')

# Create an interactive line chart
fig = px.line(df_melted,
              x='Year',
              y='Count',
              color='Crash Type',
              title='Annual Frequency of Crashes by Type (2015-2025)',
              markers=True,
              hover_name='Crash Type',
              hover_data={'Count': ':,0f'})

fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of Crashes',
    hovermode='x unified'
)

fig.show()

## Severity Breakdown by Crash Type

In [ ]:
# Severity breakdown by crash type
sev_type = (
    df_study
    .groupby(["Crash Type", "Severity"])["Case Number"]
    .count()
    .unstack("Severity")
    .fillna(0)
    .astype(int)
)

# Reorder severity levels logically
sev_order = [c for c in ["Probable Property Damage", "Probable Injury", "Fatal"]
             if c in sev_type.columns]
sev_type = sev_type[sev_order]

display(sev_type)

Severity,Probable Property Damage,Probable Injury,Fatal
Crash Type,,,
Involving Cyclist,41,1119,11
Involving Pedestrian,97,1153,47
Motor Vehicle Only,6340,49130,92
Multi-Vulnerable,0,2,0


In [ ]:
# Reset index to make 'Crash Type' a column for Plotly
sev_type_reset = sev_type.reset_index()

# Create a stacked bar chart
fig = px.bar(
    sev_type_reset,
    x='Crash Type',
    y=['Probable Property Damage', 'Probable Injury', 'Fatal'],
    title='Crash Severity Breakdown by Crash Type',
    labels={
        'Crash Type': 'Crash Type',
        'value': 'Number of Crashes',
        'variable': 'Severity'
    },
    barmode='stack',
    color_discrete_map={
        'Property Damage Only': '#636efa',
        'Probable Injury': '#ef553b',
        'Fatal': '#00cc96'
    }
)

fig.update_layout(
    xaxis_title='Crash Type',
    yaxis_title='Number of Crashes',
    legend_title='Severity',
    hovermode='x unified'
)

fig.show()


In [ ]:
sev_type_normalized = sev_type.apply(lambda x: x / x.sum(), axis=1)
sev_type_normalized_reset = sev_type_normalized.reset_index()

fig_normalized = px.bar(
    sev_type_normalized_reset,
    x='Crash Type',
    y=['Probable Property Damage', 'Probable Injury', 'Fatal'],
    title='Normalized Crash Severity Breakdown by Crash Type (100% Stacked Bar)',
    labels={
        'Crash Type': 'Crash Type',
        'value': 'Proportion of Crashes',
        'variable': 'Severity'
    },
    barmode='relative',
    color_discrete_map={
        'Property Damage Only': '#636efa',
        'Probable Injury': '#ef553b',
        'Fatal': '#00cc96'
    }
)

fig_normalized.update_layout(
    xaxis_title='Crash Type',
    yaxis_title='Proportion of Crashes',
    yaxis_tickformat='.0%',
    legend_title='Severity',
    hovermode='x unified'
)

fig_normalized.show()

In [ ]:
import scipy.stats as stats

# Create a contingency table for Chi-square test
# We'll group 'Property Damage Only' and 'Probable Injury' into 'Non-Fatal'
contingency_table = sev_type[['Probable Property Damage', 'Probable Injury', 'Fatal']].copy()
contingency_table['Non-Fatal'] = contingency_table['Probable Property Damage'] + contingency_table['Probable Injury']
contingency_table = contingency_table[['Fatal', 'Non-Fatal']]

print("Contingency Table for Chi-square Test:")
print(contingency_table)

# Perform Chi-square test for independence
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"\nChi-square Statistic: {chi2:.2f}")
print(f"P-value: {p_value:.3e}") # Display p-value in scientific notation for clarity
print(f"Degrees of Freedom: {dof}")

alpha = 0.05 # Significance level

print("\nNull Hypothesis (H0): There is no statistically significant association between Crash Type and Fatality.")

if p_value < alpha:
    print("\nConclusion: The p-value is less than the significance level (0.05), so we reject the null hypothesis.")
    print("There is a statistically significant association between Crash Type and Fatality.")
else:
    print("\nConclusion: The p-value is greater than the significance level (0.05), so we fail to reject the null hypothesis.")
    print("There is no statistically significant association between Crash Type and Fatality.")


Contingency Table for Chi-square Test:
Severity              Fatal  Non-Fatal
Crash Type                            
Involving Cyclist        11       1160
Involving Pedestrian     47       1250
Motor Vehicle Only       92      55470
Multi-Vulnerable          0          2

Chi-square Statistic: 609.41
P-value: 9.211e-132
Degrees of Freedom: 3

Null Hypothesis (H0): There is no statistically significant association between Crash Type and Fatality.

Conclusion: The p-value is less than the significance level (0.05), so we reject the null hypothesis.
There is a statistically significant association between Crash Type and Fatality.


## Logistic Regression: Crash Type and Fatality

severity of crashes of motor vehicles differ from the frequency and severity of crashes involving vulnerable users

In [ ]:
import statsmodels.api as sm

# 1. Create a binary target variable 'IsFatal'
df_study['IsFatal'] = (df_study['Severity'] == 'Fatal').astype(bool)

# 2. Convert 'Crash Type' into dummy variables, ensuring they are integers
X = pd.get_dummies(df_study['Crash Type'], prefix='CrashType', drop_first=True).astype(int)
y = df_study['IsFatal']

# 3. Add a constant to the independent variables for the intercept
X = sm.add_constant(X)

# 4. Fit the logistic regression model
logit_model = sm.Logit(y, X)
result = logit_model.fit()

# 5. Print the model summary
print(result.summary())

         Current function value: 0.016291
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:                IsFatal   No. Observations:                58032
Model:                          Logit   Df Residuals:                    58028
Method:                           MLE   Df Model:                            3
Date:                Tue, 14 Apr 2026   Pseudo R-squ.:                 0.09403
Time:                        14:19:45   Log-Likelihood:                -945.40
converged:                      False   LL-Null:                       -1043.5
Covariance Type:            nonrobust   LLR p-value:                 2.740e-42
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const                             -4.6583      0.303    -15.377      0.000      -5.252      -4.065
Crash

## Pre- and Post-COVID Period Analysis

In [ ]:
# Group by 'Period' and 'Crash Type' and count 'Case Number'
crash_type_period_counts = df_study.groupby(['Period', 'Crash Type'])['Case Number'].count().unstack(fill_value=0)

# Define the number of years for each period
years_period1 = 5 # 2015-2019
years_period2 = 6 # 2020-2025

# Normalize crash counts to crashes-per-year
normalized_counts = crash_type_period_counts.copy()
normalized_counts.loc['Period 1: 2015–2019'] = normalized_counts.loc['Period 1: 2015–2019'] / years_period1
normalized_counts.loc['Period 2: 2020–2025'] = normalized_counts.loc['Period 2: 2020–2025'] / years_period2

print("Normalized Annual Crash Counts (Crashes per Year):")
display(normalized_counts)

# Melt the DataFrame to long format for Plotly Express
df_plot = normalized_counts.reset_index().melt(id_vars='Period', var_name='Crash Type', value_name='Crashes Per Year')

# Create an interactive grouped bar chart with improved labels
fig = px.bar(df_plot,
             x='Crash Type',
             y='Crashes Per Year',
             color='Period',
             barmode='group',
             title='Comparison of Annual Crash Rates by Type: Pre-COVID (2015-2019) vs Post-COVID (2020-2025)',
             labels={'Crashes Per Year': 'Average Crashes Per Year'},
             hover_data={'Crashes Per Year': ':.1f'},
             text_auto='.1f') # Display values on bars with one decimal place

fig.update_layout(
    xaxis_title='Crash Type',
    yaxis_title='Average Crashes Per Year',
    legend_title='Analysis Period', # Add a legend title
    hovermode='x unified'
)

fig.show()


Normalized Annual Crash Counts (Crashes per Year):


Crash Type,Involving Cyclist,Involving Pedestrian,Motor Vehicle Only,Multi-Vulnerable
Period,,,,
Period 1: 2015–2019,129.2,111.8,6070,0.200000
Period 2: 2020–2025,87.5,123.0,4202,0.166667


## Crash Frequency by Hour

In [ ]:
print('Hour-of-Day Analysis')

# 1. Descriptive Statistics

# Frequency counts (crashes per hour)
hourly_crashes = df_study['Crash Hour of Day'].value_counts().sort_index()

# Percent distribution
total_crashes = hourly_crashes.sum()
hourly_crashes_percent = (hourly_crashes / total_crashes) * 100

# Combine into a single DataFrame for display
hourly_analysis = pd.DataFrame({
    'Total Crashes': hourly_crashes,
    'Percentage': hourly_crashes_percent
}).reset_index()
hourly_analysis.columns = ['Crash Hour of Day', 'Total Crashes', 'Percentage']

print("\n1. Descriptive Statistics: Crashes per Hour")
display(hourly_analysis)


# Given the request for 'frequency counts', the 'Total Crashes' for each hour already serves this purpose

# Plotting the frequency counts to visualize peaks
fig_hourly = px.bar(hourly_analysis,
                    x='Crash Hour of Day',
                    y='Total Crashes',
                    title='Total Crashes by Hour of Day (2015-2025)',
                    labels={'Crash Hour of Day': 'Hour of Day', 'Total Crashes': 'Number of Crashes'},
                    hover_data={'Percentage': ':.2f'})

fig_hourly.update_layout(
    xaxis={'tickmode': 'linear'},
    yaxis_title='Number of Crashes',
    hovermode='x unified'
)

fig_hourly.show()

Hour-of-Day Analysis

1. Descriptive Statistics: Crashes per Hour


,Crash Hour of Day,Total Crashes,Percentage
0,0,1023,1.762821
1,1,666,1.147643
2,2,666,1.147643
3,3,486,0.837469
4,4,317,0.546250
5,5,304,0.523849
6,6,429,0.739247
7,7,1087,1.873104
8,8,2706,4.662945
9,9,2406,4.145988


## Crash Severity by Hour of Day

In [ ]:
# Group by 'Crash Hour of Day' and 'Severity' to get counts
hourly_severity_counts = df_study.groupby(['Crash Hour of Day', 'Severity']).size().unstack(fill_value=0)

# Ensure all severity columns are present and in a logical order for plotting
severity_order = ['Property Damage Only', 'Probable Injury', 'Fatal']
for col in severity_order:
    if col not in hourly_severity_counts.columns:
        hourly_severity_counts[col] = 0
hourly_severity_counts = hourly_severity_counts[severity_order]

# Normalize the counts to get proportions for a 100% stacked bar chart
hourly_severity_proportions = hourly_severity_counts.apply(lambda x: x / x.sum(), axis=1)

# Reset index to make 'Crash Hour of Day' a column for Plotly
hourly_severity_proportions_reset = hourly_severity_proportions.reset_index()

# Melt the DataFrame for Plotly Express to create a stacked bar chart
df_melted_hour_severity = hourly_severity_proportions_reset.melt(
    id_vars=['Crash Hour of Day'],
    value_vars=severity_order,
    var_name='Severity',
    value_name='Proportion'
)

# Create a 100% stacked bar chart
fig_hourly_severity = px.bar(
    df_melted_hour_severity,
    x='Crash Hour of Day',
    y='Proportion',
    color='Severity',
    title='Proportion of Crash Severity by Hour of Day',
    labels={
        'Crash Hour of Day': 'Hour of Day',
        'Proportion': 'Proportion of Crashes',
        'Severity': 'Crash Severity'
    },
    barmode='stack',
    color_discrete_map={
        'Property Damage Only': '#636efa',
        'Probable Injury': '#ef553b',
        'Fatal': '#00cc96'
    }
)

fig_hourly_severity.update_layout(
    xaxis_title='Hour of Day',
    yaxis_title='Proportion of Crashes',
    yaxis_tickformat='.0%', # Format y-axis as percentage
    legend_title='Severity',
    hovermode='x unified'
)

fig_hourly_severity.show()


## Top 20 Crash Address Segments

In [ ]:
#  Top-20 crash address segments

address_crashes = df_study['Address'].value_counts().reset_index()
address_crashes.columns = ['Address', 'Total Crashes']

# Calculate total fatalities per address segment
address_fatalities = df_study.groupby('Address')['Total Fatalities'].sum().reset_index()
address_fatalities.columns = ['Address', 'Total Fatalities']

# Merge total crashes with total fatalities
address_crashes = pd.merge(address_crashes, address_fatalities, on='Address', how='left')
address_crashes['Total Fatalities'] = address_crashes['Total Fatalities'].fillna(0).astype(int)

# Filter out 'UNKNOWN' or NaN addresses if any, and then get the top 20
address_crashes = address_crashes[address_crashes['Address'] != 'UNKNOWN']
address_crashes_top20 = address_crashes.head(20).copy()

# Identify the address with the highest fatalities to highlight it
max_fatal_val = address_crashes_top20['Total Fatalities'].max()
address_crashes_top20['Highlight'] = address_crashes_top20['Total Fatalities'].apply(
    lambda x: 'Highest Fatality' if x == max_fatal_val and x > 0 else 'Other'
)

fig_top20_addresses = px.bar(address_crashes_top20,
                            x='Total Crashes',
                            y='Address',
                            orientation='h',
                            color='Highlight',
                            color_discrete_map={'Highest Fatality': 'red', 'Other': '#636efa'},
                            title='Top 20 Crash Address Segments (Highlighted by Highest Fatality)',
                            labels={'Total Crashes': 'Number of Crashes', 'Address': 'Street Address'},
                            hover_data={'Total Crashes': ':,0f', 'Total Fatalities': ':,0f', 'Highlight': False})

fig_top20_addresses.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_title='Number of Crashes',
    yaxis_title='Address Segment',
    hovermode='y unified',
    showlegend=False
)

fig_top20_addresses.show()

## Intersection-Type Breakdown

In [ ]:
#  Intersection Type Breakdown

# Count crashes by Intersect Type
intersection_type_counts = df_study['Intersect Type'].value_counts(dropna=False).reset_index()
intersection_type_counts.columns = ['Intersect Type', 'Total Crashes']

# Rename NaN to 'Unknown' for better readability in plots
intersection_type_counts['Intersect Type'] = intersection_type_counts['Intersect Type'].fillna('Unknown')

print("Crash Counts by Intersection Type:")
display(intersection_type_counts)

# Create a bar chart for intersection type breakdown
fig_intersect_type = px.bar(intersection_type_counts,
                            x='Total Crashes',
                            y='Intersect Type',
                            orientation='h',
                            title='Distribution of Crashes by Intersection Type (2015-2025)',
                            labels={'Total Crashes': 'Number of Crashes', 'Intersect Type': 'Intersection Type'},
                            hover_data={'Total Crashes': ':,0f'})

fig_intersect_type.update_layout(
    yaxis={'categoryorder':'total ascending'},
    xaxis_title='Number of Crashes',
    yaxis_title='Intersection Type',
    hovermode='y unified'
)

fig_intersect_type.show()

Crash Counts by Intersection Type:


,Intersect Type,Total Crashes
0,NOT AT INTERSECTION,37482
1,FOUR-WAY INTERSECTION,12933
2,T-INTERSECTION,5201
3,OTHER (EXPLAIN IN NARRATIVE),1253
4,ROUNDABOUT,503
5,Y-INTERSECTION,494
6,TRAFFIC CIRCLE,139
7,"FIVE-POINT, OR MORE",21
8,Unknown,6


## Logistic Regression: Intersection Characteristics and Fatality
Severity of crash based on intersection type (complete time period )

In [ ]:
# Filter out 'Unknown' intersection types for the analysis
df_filtered = df_study[df_study['Intersect Type'].notna() & (df_study['Intersect Type'] != 'Unknown')].copy()

# 1. Create a binary target variable 'IsFatal'
# Ensure it's of integer type for logistic regression
y = (df_filtered['Severity'] == 'Fatal').astype(int)

# 2. Convert 'Intersect Type' into dummy variables, ensuring they are integers
X = pd.get_dummies(df_filtered['Intersect Type'], prefix='IntersectType', drop_first=True).astype(int)

# 3. Add a constant to the independent variables for the intercept
X = sm.add_constant(X)

# 4. Fit the logistic regression model
logit_model = sm.Logit(y, X)
result = logit_model.fit()

# 5. Print the model summary
print(result.summary())

         Current function value: 0.017882
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:               Severity   No. Observations:                58026
Model:                          Logit   Df Residuals:                    58018
Method:                           MLE   Df Model:                            7
Date:                Tue, 14 Apr 2026   Pseudo R-squ.:                0.005668
Time:                        14:20:10   Log-Likelihood:                -1037.6
converged:                      False   LL-Null:                       -1043.5
Covariance Type:            nonrobust   LLR p-value:                    0.1063
                                                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------
const                                         -2.9957      1.025     -2.924     

Regression Analysis Idea   
Y = severity ( fatality ), x = crash type and intersection (not at intersection vs intersection)

Differences in differences for crash type and severity pre and post covid

## Crash Type and Fatality Before and After COVID-19

In [ ]:
import statsmodels.api as sm
import pandas as pd

# --- 1. Logistic Regression ---
# Y = fatality, X = crash type and intersection (not at intersection vs intersection)

# Create binary variables
df_study['IsFatal'] = (df_study['Severity'] == 'Fatal').astype(int)
df_study['AtIntersection'] = df_study['Intersect Type'].apply(lambda x: 0 if x == 'NOT AT INTERSECTION' else 1)

# Filter out 'Multi-Vulnerable' due to low counts causing instability
df_lr = df_study[df_study['Crash Type'] != 'Multi-Vulnerable'].copy()

# Prepare features (dummies for Crash Type + Intersection flag)
X_lr = pd.get_dummies(df_lr['Crash Type'], prefix='Type', drop_first=True).astype(int)
X_lr['AtIntersection'] = df_lr['AtIntersection']
X_lr = sm.add_constant(X_lr)
y_lr = df_lr['IsFatal']

logit_model = sm.Logit(y_lr, X_lr)
logit_res = logit_model.fit()

print("=== LOGISTIC REGRESSION RESULTS ===")
print(logit_res.summary())

# --- 2. Differences-in-Differences ---
# Crash type and severity pre and post covid

# Define periods
years_pre = 5 # 2015-2019
years_post = 6 # 2020-2025

did_summary = df_study.groupby(['Period', 'Crash Type']).agg(
    Total_Crashes=('Case Number', 'count'),
    Fatalities=('IsFatal', 'sum')
).reset_index()

# Calculate normalized annual rates
def normalize_rate(row):
    years = years_pre if "Period 1" in row['Period'] else years_post
    return row['Total_Crashes'] / years

did_summary['Avg_Annual_Crashes'] = did_summary.apply(normalize_rate, axis=1)
did_summary['Fatality_Rate'] = did_summary['Fatalities'] / did_summary['Total_Crashes']

# Pivot for comparison
pivot_did = did_summary.pivot(index='Crash Type', columns='Period', values=['Avg_Annual_Crashes', 'Fatality_Rate'])

# Calculate the Difference (Post - Pre)
pivot_did[('Avg_Annual_Crashes', 'Change')] = pivot_did[('Avg_Annual_Crashes', 'Period 2: 2020–2025')] - pivot_did[('Avg_Annual_Crashes', 'Period 1: 2015–2019')]
pivot_did[('Fatality_Rate', 'Change')] = pivot_did[('Fatality_Rate', 'Period 2: 2020–2025')] - pivot_did[('Fatality_Rate', 'Period 1: 2015–2019')]

print("\n=== DIFFERENCES-IN-DIFFERENCES SUMMARY ===")
display(pivot_did.round(4))

Optimization terminated successfully.
         Current function value: 0.016284
         Iterations 10
=== LOGISTIC REGRESSION RESULTS ===
                           Logit Regression Results                           
Dep. Variable:                IsFatal   No. Observations:                58030
Model:                          Logit   Df Residuals:                    58026
Method:                           MLE   Df Model:                            3
Date:                Tue, 14 Apr 2026   Pseudo R-squ.:                 0.09447
Time:                        14:20:13   Log-Likelihood:                -944.94
converged:                       True   LL-Null:                       -1043.5
Covariance Type:            nonrobust   LLR p-value:                 1.728e-42
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        -4.7655     

Avg_Annual_Crashes                      \
Period               Period 1: 2015–2019 Period 2: 2020–2025   
Crash Type                                                     
Involving Cyclist                  129.2             87.5000   
Involving Pedestrian               111.8            123.0000   
Motor Vehicle Only                6070.0           4202.0000   
Multi-Vulnerable                     0.2              0.1667   

                           Fatality_Rate                      \
Period               Period 1: 2015–2019 Period 2: 2020–2025   
Crash Type                                                     
Involving Cyclist                 0.0046              0.0152   
Involving Pedestrian              0.0447              0.0298   
Motor Vehicle Only                0.0014              0.0020   
Multi-Vulnerable                  0.0000              0.0000   

                     Avg_Annual_Crashes Fatality_Rate  
Period                           Change        Change  
Crash Type                                             
Involving Cyclist              -41.7000        0.0106  
Involving Pedestrian            11.2000       -0.0149  
Motor Vehicle Only           -1868.0000        0.0007  
Multi-Vulnerable                -0.0333        0.0000

Interpretation:
  - Motor Vehicle Only: The number of total crashes decreased significantly post-COVID. Fatal crashes also decreased, but the proportion of fatal crashes slightly increased, indicating a relatively higher severity for this crash type.
  - Involving Cyclist: Both total and fatal crashes decreased, and the proportion of fatal crashes also saw a decrease.
  - Involving Pedestrian: Notably, both total crashes and fatal crashes increased post-COVID, although the proportion of fatal crashes slightly decreased.
  - Multi-Vulnerable: Due to extremely low counts (only 2 total crashes), the results for this category are not statistically reliable or meaningful for this analysis.



### Difference-in-Differences (DiD) Regression Analysis

To formally test the statistical significance of the observed changes in fatality rates between the pre- and post-COVID periods for different crash types, we will employ a Difference-in-Differences (DiD) regression model.

We will use a logistic regression since our outcome variable, `IsFatal`, is binary (0 for non-fatal, 1 for fatal).

The model will include:
- `Post`: A binary variable indicating the period (0 for 2015-2019 (Pre-COVID), 1 for 2020-2025 (Post-COVID)).
- `IsVulnerableCrash`: A binary variable indicating if the crash involves a vulnerable road user (1 for 'Involving Cyclist' or 'Involving Pedestrian', 0 for 'Motor Vehicle Only').
- `Interaction (Post * IsVulnerableCrash)`: The key DiD term, capturing the differential change in fatality rates for vulnerable crash types in the post-COVID period compared to motor vehicle only crashes.

`Multi-Vulnerable` crashes will be excluded due to their extremely low frequency, which can lead to unstable regression estimates.

In [ ]:
import statsmodels.api as sm

# Prepare the data for DiD logistic regression
# 1. Create 'Post' period indicator
df_study['Post'] = (df_study['Year'] >= 2020).astype(int)

# 2. Create 'IsVulnerableCrash' indicator
df_did = df_study[df_study['Crash Type'].isin(['Involving Cyclist', 'Involving Pedestrian', 'Motor Vehicle Only'])].copy()
df_did['IsVulnerableCrash'] = df_did['Crash Type'].apply(lambda x: 1 if x in ['Involving Cyclist', 'Involving Pedestrian'] else 0)

# 3. Define the dependent variable (already exists as 'IsFatal')
y_did = df_did['IsFatal']

# 4. Define independent variables
X_did = df_did[['Post', 'IsVulnerableCrash']].copy()
X_did['Interaction'] = X_did['Post'] * X_did['IsVulnerableCrash']
X_did = sm.add_constant(X_did)

# 5. Fit the logistic regression model
did_logit_model = sm.Logit(y_did, X_did)
did_logit_result = did_logit_model.fit()

# 6. Print the model summary
print("=== DIFFERENCE-IN-DIFFERENCES LOGISTIC REGRESSION RESULTS ===")
print(did_logit_result.summary())

# Interpret the DiD coefficient (Interaction term)
interaction_coef = did_logit_result.params['Interaction']
interaction_pvalue = did_logit_result.pvalues['Interaction']

print(f"\nDiD (Interaction) Coefficient: {interaction_coef:.4f}")
print(f"DiD (Interaction) P-value: {interaction_pvalue:.4f}")

alpha = 0.05
if interaction_pvalue < alpha:
    print("Conclusion: The DiD interaction term is statistically significant at the 5% level.")
    print("This suggests a statistically significant differential change in fatality likelihood for vulnerable crash types post-COVID compared to motor vehicle only crashes.")
else:
    print("Conclusion: The DiD interaction term is NOT statistically significant at the 5% level.")
    print("This suggests no statistically significant differential change in fatality likelihood for vulnerable crash types post-COVID compared to motor vehicle only crashes.")

Optimization terminated successfully.
         Current function value: 0.016440
         Iterations 11
=== DIFFERENCE-IN-DIFFERENCES LOGISTIC REGRESSION RESULTS ===
                           Logit Regression Results                           
Dep. Variable:                IsFatal   No. Observations:                58030
Model:                          Logit   Df Residuals:                    58026
Method:                           MLE   Df Model:                            3
Date:                Tue, 14 Apr 2026   Pseudo R-squ.:                 0.08576
Time:                        14:28:31   Log-Likelihood:                -954.02
converged:                       True   LL-Null:                       -1043.5
Covariance Type:            nonrobust   LLR p-value:                 1.459e-38
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -6.6056   

## Difference-in-Differences (DiD) Analysis: Impact on Fatal Crashes

### Objective:
To assess if the Post-COVID period (2020-2025) had a statistically significant *differential impact* on fatality likelihood for vulnerable road user crashes (cyclists/pedestrians) compared to motor vehicle only crashes, using a logistic regression model.

### Key Variables:
*   **Dependent Variable:** `IsFatal` (1 = Fatal Crash, 0 = Non-Fatal Crash)
*   **Treatment Group:** `IsVulnerableCrash` (1 = Involving Cyclist/Pedestrian, 0 = Motor Vehicle Only)
*   **Post-Intervention Period:** `Post` (1 = 2020-2025, 0 = 2015-2019)
*   **DiD Term:** `Interaction` (`Post * IsVulnerableCrash`)

### Regression Results Summary:
| Term                | Coefficient | P-value | Interpretation                                                                                                                                                                                                  |
|---------------------|-------------|---------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `IsVulnerableCrash` | **+2.8671** | **0.000** | **Highly Significant:** In the Pre-COVID period, crashes involving vulnerable road users had significantly higher odds of being fatal compared to motor vehicle only crashes.                                    |
| `Post`              | +0.4044     | 0.054   | **Not Statistically Significant (at α=0.05):** There was no statistically significant *overall* change in fatality odds for **Motor Vehicle Only** crashes from Pre- to Post-COVID.                                    |
| `Interaction` (DiD) | -0.3819     | 0.260   | **Not Statistically Significant:** There is **no statistically significant differential change** in fatality odds for vulnerable crash types in the Post-COVID period, compared to motor vehicle only crashes. |

### Conclusion for Slides:

*   **Vulnerable road users (cyclists/pedestrians) are at a consistently and significantly higher risk of fatal outcomes in crashes compared to motor vehicle only incidents (p < 0.001).**
*   **While there were shifts in overall crash trends post-COVID, the Difference-in-Differences analysis indicates that the post-COVID period did *not* lead to a statistically significant *change in the relative risk of fatality* for vulnerable users compared to motor vehicle only occupants (p = 0.260).**
*   This suggests that the factors influencing fatal crash likelihood for vulnerable users, relative to motor vehicles, remained largely consistent across the pre- and post-COVID periods, despite potential changes in overall crash frequencies.

In [ ]:
# Define periods for normalization
years_pre = 5  # 2015-2019
years_post = 6 # 2020-2025

# Group by 'Period' and 'Crash Type' and aggregate total crashes and fatalities
did_summary = df_study.groupby(['Period', 'Crash Type']).agg(
    Total_Crashes=('Case Number', 'count'),
    Fatalities=('IsFatal', 'sum')
).reset_index()

# Calculate normalized annual rates
def normalize_rate(row):
    years = years_pre if "Period 1" in row['Period'] else years_post
    return row['Total_Crashes'] / years

did_summary['Avg_Annual_Crashes'] = did_summary.apply(normalize_rate, axis=1)
did_summary['Fatality_Rate'] = did_summary['Fatalities'] / did_summary['Total_Crashes']

# Pivot the table to compare Pre- and Post-COVID periods side-by-side
pivot_did = did_summary.pivot(index='Crash Type', columns='Period', values=['Avg_Annual_Crashes', 'Fatality_Rate'])

# Calculate the Difference (Post - Pre) for both average annual crashes and fatality rates
pivot_did[('Avg_Annual_Crashes', 'Change')] = pivot_did[('Avg_Annual_Crashes', 'Period 2: 2020–2025')] - pivot_did[('Avg_Annual_Crashes', 'Period 1: 2015–2019')]
pivot_did[('Fatality_Rate', 'Change')] = pivot_did[('Fatality_Rate', 'Period 2: 2020–2025')] - pivot_did[('Fatality_Rate', 'Period 1: 2015–2019')]

print("=== DIFFERENCES-IN-DIFFERENCES DESCRIPTIVE SUMMARY ===")
display(pivot_did.round(4))


=== DIFFERENCES-IN-DIFFERENCES DESCRIPTIVE SUMMARY ===


Avg_Annual_Crashes                      \
Period               Period 1: 2015–2019 Period 2: 2020–2025   
Crash Type                                                     
Involving Cyclist                  129.2             87.5000   
Involving Pedestrian               111.8            123.0000   
Motor Vehicle Only                6070.0           4202.0000   
Multi-Vulnerable                     0.2              0.1667   

                           Fatality_Rate                      \
Period               Period 1: 2015–2019 Period 2: 2020–2025   
Crash Type                                                     
Involving Cyclist                 0.0046              0.0152   
Involving Pedestrian              0.0447              0.0298   
Motor Vehicle Only                0.0014              0.0020   
Multi-Vulnerable                  0.0000              0.0000   

                     Avg_Annual_Crashes Fatality_Rate  
Period                           Change        Change  
Crash Type                                             
Involving Cyclist              -41.7000        0.0106  
Involving Pedestrian            11.2000       -0.0149  
Motor Vehicle Only           -1868.0000        0.0007  
Multi-Vulnerable                -0.0333        0.0000